# VRC-3PO revision — R1b: the environment shortcut

**Question:** once the model can no longer recognise the VR environment, is there still an eye-tracking signal for cybersickness in people it has never seen?

**Design (pre-specified; every variant is reported, not just the best):**
* Same seated data (47 participants, benchmark v1.1), same 5-fold participant CV, 3 repeats, identical folds for every variant.
* All variants are scored on the **same windows**: those starting at second ≥ 60, so the per-user calibration minute is never evaluated.
* Input variants = channel set × calibration

| channels | what it removes |
|---|---|
| `all` | nothing (published input) |
| `no_origin` | world-space gaze origin: a location tag that identifies the scene 94 % of the time |
| `pupil` | everything except pupil diameter |
| `pupil_head` | everything except pupil diameter and head orientation |

`cal60` = every channel z-scored by the person's own first 60 s of that recording (clipped at ±10 SD): deployable as a one-minute baseline at session start. `raw` = the published inputs.
* Metrics: pooled AUC; **within-environment AUC** (same environment and CV fold only: exactly 0.5 for any environment-only score); **within-environment-and-minute AUC** (also the same minute of exposure, so time-on-task cannot explain it); incremental AUC over environment identity and over environment + elapsed time.

Runtime on a T4: Steps A–C ≈ 10 min (CPU), Step D ≈ 2.5–3.5 h (resumable; the summary can be run at any time on finished repeats).

In [ ]:
# ═══ 1 · Runtime ═══
!pip install -q keras-tcn
import tensorflow as tf, pandas as pd, numpy as np
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ═══ 2 · Drive + code (bundle v2) ═══
from google.colab import drive, files
from pathlib import Path
import zipfile, shutil, os
drive.mount('/content/drive')
REV = Path('/content/drive/MyDrive/vrc3po_revision')
BUNDLE = REV / 'vrc3po_revision_bundle_v2.zip'
if not BUNDLE.exists():
    print('Upload vrc3po_revision_bundle_v2.zip from the repo root on your Mac:')
    up = files.upload()
    shutil.copy([n for n in up if n.endswith('.zip')][0], BUNDLE)
root = Path('/content/vrc3po_code'); shutil.rmtree(root, ignore_errors=True); root.mkdir()
zipfile.ZipFile(BUNDLE).extractall(root); os.chdir(root)
assert 'calibrate_per_recording' in (root / 'analysis/revision_common.py').read_text(), 'wrong bundle'
!pip install -q pytest
!python -m pytest tests -q

In [ ]:
# ═══ 3 · Paths ═══
V11 = REV / 'vrc3po_benchmark_v1_1.csv'
OUT = REV / 'cv'                     # same run store as before; new runs have their own scope tags
assert V11.exists(), 'benchmark v1.1 missing'
SCOPES = ['seated-all-raw-s60', 'seated-pupil-cal60-s60', 'seated-no_origin-cal60-s60',
          'seated-all-cal60-s60', 'seated-pupil_head-cal60-s60', 'seated-pupil-raw-s60',
          'seated-no_origin-raw-s60']    # priority order
print('ok')

## Step A · Which channels give the scene away? (CPU, ~3 min)

In [ ]:
!python -m analysis.environment_probe --dataset "$V11" --output "$OUT/environment_probe.csv"

## Step B · Sensor-free baselines on the evaluation windows (CPU, ~1 min)
Environment identity, elapsed time, and both together: everything an application knows without measuring the user.

In [ ]:
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --scope seated-all-raw-s60 --models nuisance:condition nuisance:clock nuisance:condition_clock --repeats 3 --folds 5

## Step C · Feature-based detectors on every input variant (CPU, ~5 min)

In [ ]:
for sc in SCOPES:
    !python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --scope {sc} --models feat:logreg feat:rf --repeats 3 --folds 5

## Step D · CNN ensemble (3 members) on every input variant (GPU, ~2.5–3.5 h, resumable)
Variants run in priority order; if time runs out, the first three already answer the question.

In [ ]:
assert tf.config.list_physical_devices('GPU'), 'Switch to a GPU runtime.'
for sc in SCOPES:
    !python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --scope {sc} --models cnn_ensemble --members 42 123 456 --repeats 3 --folds 5

## Step E · Summary → `results_revision_r1b.zip` on Drive (can be run on partial results)

In [ ]:
import subprocess, json
proc = subprocess.Popen(['python', '-m', 'analysis.revision_cv', 'summarize', '--dataset', str(V11),
                         '--output-dir', str(OUT), '--bootstrap', '2000'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if line.startswith('['): print(line, end='')
assert proc.wait() == 0, 'summary failed'
S = OUT / 'summary'
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
print('\nWHICH CHANNELS REVEAL THE SCENE'); display(pd.read_csv(OUT / 'environment_probe.csv').pivot(index='channels', columns='calibration', values='environment_accuracy').round(3))
v = pd.read_csv(S / 'input_variants.csv')
print('\nINPUT VARIANTS (paired vs all-channels raw, same windows & folds)')
display(v[['model','channels','calibration','repeats','pooled_auc','within_condition_auc','within_condition_delta','within_condition_delta_ci_low','within_condition_delta_ci_high','within_condition_minute_auc']].round(3))
p = pd.read_csv(S / 'primary_estimates.csv'); p = p[p.scope.str.contains('-s60')]
print('\nPER-VARIANT ESTIMATES WITH CIs')
display(p[['scope','model','repeats','auc_mean','auc_ci_low','auc_ci_high','within_condition_auc','within_condition_ci_low','within_condition_ci_high','within_condition_minute_auc','within_condition_minute_ci_low','within_condition_minute_ci_high']].round(3))
e = pd.read_csv(S / 'environment_control.csv'); e = e[e.scope.str.contains('-s60')]
print('\nADDED VALUE OVER SENSOR-FREE BASELINES')
display(e[['scope','model','auc_environment_only','increment','increment_ci_low','increment_ci_high','auc_environment_clock','increment_over_env_clock','increment_env_clock_ci_low','increment_env_clock_ci_high']].round(3))

bundle = Path('/content/results_revision_r1b'); shutil.rmtree(bundle, ignore_errors=True); bundle.mkdir()
shutil.copytree(S, bundle / 'cv_summary'); shutil.copy(OUT / 'environment_probe.csv', bundle)
shutil.make_archive('/content/results_revision_r1b', 'zip', bundle)
shutil.copy('/content/results_revision_r1b.zip', REV / 'results_revision_r1b.zip')
os.chdir('/content'); drive.flush_and_unmount()
print('✓ MyDrive/vrc3po_revision/results_revision_r1b.zip — copy it into VRC-3PO/ on your Mac')